In [1]:
import yaml
import pandas as pd

In [2]:
with open("Price_List_table.yaml", "r") as file:
    data = yaml.safe_load(file)

In [3]:
df=pd.DataFrame(data)

In [4]:
df.head()

,MAP,PL,sku
0,106684,DELL0X_PC_Laptop,0X749QB
1,35611,DELL75_PC_Monitor,75X62BB
2,29914,DELLDF_SUP_TON,DF627B
3,39258,HP4Z_PH_LJ,4ZB96A
4,139670,DELL0R_PC_Laptop,0R0N0QB


In [5]:
lpp_df=pd.read_csv("LPP_table.csv")

In [6]:
lpp_df.head(5)

,Sub_category,Hp,Dell
0,Laptop,0.034,0.078
1,Monitor,0.012,0.056
2,Toner,0.091,0.023
3,Laserjet,0.065,0.035
4,Inkjet,0.012,0.056


In [7]:
df.head(5)

,MAP,PL,sku
0,106684,DELL0X_PC_Laptop,0X749QB
1,35611,DELL75_PC_Monitor,75X62BB
2,29914,DELLDF_SUP_TON,DF627B
3,39258,HP4Z_PH_LJ,4ZB96A
4,139670,DELL0R_PC_Laptop,0R0N0QB


In [8]:
subcategory_df=pd.read_csv("PL_Table_Cleaned.csv")

In [9]:
subcategory_df.head(5)

,sku,PL,subcategory
0,0X749QB,DELL0X_PC_Laptop,Laptop
1,75X62BB,DELL75_PC_Monitor,Monitor
2,DF627B,DELLDF_SUP_TON,Toner
3,4ZB96A,HP4Z_PH_LJ,Laserjet
4,0R0N0QB,DELL0R_PC_Laptop,Laptop


In [10]:
merged_df = pd.merge(df, subcategory_df, on=["sku","PL"], how="inner")

In [11]:
merged_df.head(5)

,MAP,PL,sku,subcategory
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop
1,35611,DELL75_PC_Monitor,75X62BB,Monitor
2,29914,DELLDF_SUP_TON,DF627B,Toner
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop


In [12]:
merged_df.isna().sum()

MAP            0
PL             0
sku            0
subcategory    0
dtype: int64

In [13]:
merged_df["brand"] = merged_df["PL"].str.extract(r'^(HP|DELL)', expand=False)

In [14]:
merged_df["brand"] = merged_df["brand"].str.capitalize()

In [15]:
merged_df.head(5)

,MAP,PL,sku,subcategory,brand
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop,Dell
1,35611,DELL75_PC_Monitor,75X62BB,Monitor,Dell
2,29914,DELLDF_SUP_TON,DF627B,Toner,Dell
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet,Hp
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop,Dell


In [16]:
lpp_long = lpp_df.melt(id_vars="Sub_category",var_name="brand",value_name="lpp_percentage")

In [17]:
lpp_long.head(5)

,Sub_category,brand,lpp_percentage
0,Laptop,Hp,0.034
1,Monitor,Hp,0.012
2,Toner,Hp,0.091
3,Laserjet,Hp,0.065
4,Inkjet,Hp,0.012


In [18]:
final_df = pd.merge(merged_df,lpp_long,left_on=["subcategory","brand"],right_on=["Sub_category","brand"],how="left")

In [19]:
final_df.head(5)

,MAP,PL,sku,subcategory,brand,Sub_category,lpp_percentage
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop,Dell,Laptop,0.078
1,35611,DELL75_PC_Monitor,75X62BB,Monitor,Dell,Monitor,0.056
2,29914,DELLDF_SUP_TON,DF627B,Toner,Dell,Toner,0.023
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet,Hp,Laserjet,0.065
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop,Dell,Laptop,0.078


In [20]:
(final_df["subcategory"] == final_df["Sub_category"]).all()

np.False_

In [21]:
final_df[final_df["subcategory"] != final_df["Sub_category"]][
    ["sku","subcategory","Sub_category"]
]

,sku,subcategory,Sub_category
18,L0R91AA,Unknown,NaN
94,7AE72BB,Unknown,NaN
115,31H19B,SJ,NaN
206,7GX17B,SJ,NaN
235,K4N82B,Unknown,NaN
289,J3M68A,Unknown,NaN
342,M1S05BB,Unknown,NaN
370,7GX10B,SJ,NaN
372,L0S06AA,Unknown,NaN
424,31H17B,SJ,NaN


In [22]:
final_df = final_df.drop(columns=["Sub_category"])

In [23]:
final_df.isna().sum()

MAP                0
PL                 0
sku                0
subcategory        0
brand              0
lpp_percentage    48
dtype: int64

In [24]:
final_df.head(10)

,MAP,PL,sku,subcategory,brand,lpp_percentage
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop,Dell,0.078
1,35611,DELL75_PC_Monitor,75X62BB,Monitor,Dell,0.056
2,29914,DELLDF_SUP_TON,DF627B,Toner,Dell,0.023
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet,Hp,0.065
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop,Dell,0.078
5,993854,DELL7L_PH_IJ,7LE37B,Inkjet,Dell,0.056
6,49687,DELL89_PC_Desktop,89Z55QB,Desktop,Dell,0.023
7,102079,HP8C_PC_Laptop,8C4R9PA,Laptop,Hp,0.034
8,9900,DELLG0_SUP_INK,G0L15B,Ink,Dell,0.035
9,28788,DELLDF_SUP_TON,DF853B,Toner,Dell,0.023


In [25]:
final_df["LPP"] = final_df["MAP"] - (final_df["MAP"] * final_df["lpp_percentage"])

In [26]:
final_df.head(10)

,MAP,PL,sku,subcategory,brand,lpp_percentage,LPP
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop,Dell,0.078,98362.648
1,35611,DELL75_PC_Monitor,75X62BB,Monitor,Dell,0.056,33616.784
2,29914,DELLDF_SUP_TON,DF627B,Toner,Dell,0.023,29225.978
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet,Hp,0.065,36706.230
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop,Dell,0.078,128775.740
5,993854,DELL7L_PH_IJ,7LE37B,Inkjet,Dell,0.056,938198.176
6,49687,DELL89_PC_Desktop,89Z55QB,Desktop,Dell,0.023,48544.199
7,102079,HP8C_PC_Laptop,8C4R9PA,Laptop,Hp,0.034,98608.314
8,9900,DELLG0_SUP_INK,G0L15B,Ink,Dell,0.035,9553.500
9,28788,DELLDF_SUP_TON,DF853B,Toner,Dell,0.023,28125.876


In [27]:
final_df["LPP"] = final_df["LPP"].fillna(final_df["MAP"])

In [28]:
final_df.head(10)

,MAP,PL,sku,subcategory,brand,lpp_percentage,LPP
0,106684,DELL0X_PC_Laptop,0X749QB,Laptop,Dell,0.078,98362.648
1,35611,DELL75_PC_Monitor,75X62BB,Monitor,Dell,0.056,33616.784
2,29914,DELLDF_SUP_TON,DF627B,Toner,Dell,0.023,29225.978
3,39258,HP4Z_PH_LJ,4ZB96A,Laserjet,Hp,0.065,36706.230
4,139670,DELL0R_PC_Laptop,0R0N0QB,Laptop,Dell,0.078,128775.740
5,993854,DELL7L_PH_IJ,7LE37B,Inkjet,Dell,0.056,938198.176
6,49687,DELL89_PC_Desktop,89Z55QB,Desktop,Dell,0.023,48544.199
7,102079,HP8C_PC_Laptop,8C4R9PA,Laptop,Hp,0.034,98608.314
8,9900,DELLG0_SUP_INK,G0L15B,Ink,Dell,0.035,9553.500
9,28788,DELLDF_SUP_TON,DF853B,Toner,Dell,0.023,28125.876


In [29]:
Price_list_table_cleaned = final_df[["sku","PL","MAP","LPP"]]

In [30]:
Price_list_table_cleaned.head()

,sku,PL,MAP,LPP
0,0X749QB,DELL0X_PC_Laptop,106684,98362.648
1,75X62BB,DELL75_PC_Monitor,35611,33616.784
2,DF627B,DELLDF_SUP_TON,29914,29225.978
3,4ZB96A,HP4Z_PH_LJ,39258,36706.230
4,0R0N0QB,DELL0R_PC_Laptop,139670,128775.740


In [31]:
Price_list_table_cleaned.to_csv("Price_list_table_cleaned.csv", index=False)